> **Archived precursor.** This exploratory notebook predates the tested trajectory-anomaly lifecycle. It is retained for project history, not as the prescribed or current implementation.

# Week 3 — LSTM Autoencoder Training

**Inputs:** `data/processed/train.parquet`, `val.parquet`

**Outputs:** `models/lstm_ae.pt` — saved weights

**Deliverable:** Loss curve plotted, weights saved. Demo wired to real scores.

**Hard stop (Saturday Week 3):** If val loss is not decreasing by Saturday, ship IF-only demo.
Do not carry a broken training loop into Week 4.

**Colab:** Runtime → Change runtime type → T4 GPU. Training ~15 min on T4 vs ~2h CPU.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/drone-ai-saturdays'
    !pip install -q torch pandas pyarrow
else:
    BASE = '..'

import os
DATA_PROC = f'{BASE}/data/processed'
MODELS = f'{BASE}/models'
os.makedirs(MODELS, exist_ok=True)

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

FEATURE_COLS = ['lat','lon','alt','speed','heading','dist_lemd','tod_sin','tod_cos']
WINDOW = 30     # time steps per sequence
HIDDEN = 64
N_LAYERS = 2
BATCH = 64
EPOCHS = 50
LR = 1e-3

In [ ]:
class TrajectoryDataset(Dataset):
    def __init__(self, parquet_path, window=WINDOW):
        df = pd.read_parquet(parquet_path)
        self.sequences = []
        for seg_id, grp in df.groupby('seg_id'):
            vals = grp[FEATURE_COLS].values.astype(np.float32)
            # Sliding window over the trajectory
            for i in range(0, len(vals) - window + 1, window // 2):
                self.sequences.append(vals[i:i+window])
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx])

train_ds = TrajectoryDataset(f'{DATA_PROC}/train.parquet')
val_ds   = TrajectoryDataset(f'{DATA_PROC}/val.parquet')
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH)
print(f'Train sequences: {len(train_ds):,}, Val sequences: {len(val_ds):,}')

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)
    
    def forward(self, x):
        # Encode
        _, (h, c) = self.encoder(x)
        # Decode: repeat latent vector T times
        latent = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        decoded, _ = self.decoder(latent, (h, c))
        return self.output_layer(decoded)
    
    def anomaly_score(self, x):
        """MSE reconstruction error per sequence."""
        with torch.no_grad():
            recon = self(x)
            return ((x - recon) ** 2).mean(dim=(1, 2))

model = LSTMAutoencoder(len(FEATURE_COLS), HIDDEN, N_LAYERS).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            val_loss += criterion(model(batch), batch).item()
    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), f'{MODELS}/lstm_ae.pt')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | train: {train_loss:.4f} | val: {val_loss:.4f}')

print(f'Best val loss: {best_val:.4f} — weights saved to {MODELS}/lstm_ae.pt')

In [ ]:
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.title('LSTM Autoencoder training')
plt.legend()
plt.savefig(f'{BASE}/docs/weekly/figures/week3_loss_curve.png', dpi=100, bbox_inches='tight')
plt.show()

# Convergence check
if val_losses[-1] > val_losses[0] * 0.9:
    print('WARNING: val loss barely moved — training may not have converged.')
    print('Consider: lower LR, more epochs, or check data preprocessing.')
    print('HARD STOP decision: if this is Saturday Week 3, ship IF-only demo now.')
else:
    print(f'Loss dropped {100*(1 - val_losses[-1]/val_losses[0]):.1f}% — looks like convergence.')

In [ ]:
# Set anomaly threshold at 95th percentile of val reconstruction error
model.load_state_dict(torch.load(f'{MODELS}/lstm_ae.pt', map_location=DEVICE))
model.eval()

val_scores = []
with torch.no_grad():
    for batch in val_loader:
        scores = model.anomaly_score(batch.to(DEVICE))
        val_scores.extend(scores.cpu().numpy())

threshold = np.percentile(val_scores, 95)
print(f'Anomaly threshold (95th pct): {threshold:.4f}')

# Save threshold
np.save(f'{MODELS}/threshold.npy', threshold)

plt.hist(val_scores, bins=50)
plt.axvline(threshold, color='red', label=f'95th pct = {threshold:.4f}')
plt.xlabel('Reconstruction error (MSE)')
plt.title('Val set anomaly score distribution')
plt.legend()
plt.show()